# Parte 1 - Baseline RAG funcional

## Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña

**Implementa el pipeline sobre tu corpus, a partir del Lab-02 del día miércoles:**

In [19]:
import socket
import sys
from pathlib import Path
import sys
import os
from rag_pipeline import RagPipeline
from qdrant_client import QdrantClient
import pandas as pd


In [11]:

# Buscar la raíz del proyecto
raiz = Path.cwd()

while raiz != raiz.parent:
    if (
        (raiz / "pyproject.toml").exists()
        and (raiz / "rag_pipeline.py").exists()
    ):
        break
    raiz = raiz.parent

ROOT = raiz

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)
print("rag_pipeline.py:", (ROOT / "rag_pipeline.py").exists())
print("pyproject.toml:", (ROOT / "pyproject.toml").exists())

Raíz del proyecto: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena
rag_pipeline.py: True
pyproject.toml: True


In [12]:
# Configuración de la Parte 1

os.environ["EMBEDDING_BACKEND"] = "h200"

# Qdrant NO debe ser :memory: en la Parte 1
os.environ["QDRANT_URL"] = "http://localhost:6333"

# Colección específica del baseline
os.environ["QDRANT_COLLECTION"] = "mmia6013_rag_baseline"

# Corpus real
os.environ["CORPUS_DIR"] = str(ROOT / "corpus")

print("EMBEDDING_BACKEND:", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL:", os.environ["QDRANT_URL"])
print("CORPUS_DIR:", os.environ["CORPUS_DIR"])

EMBEDDING_BACKEND: h200
QDRANT_URL: http://localhost:6333
CORPUS_DIR: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/corpus


In [13]:


cliente_qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"]
)

print("Colecciones existentes:")
print(cliente_qdrant.get_collections())

Colecciones existentes:
collections=[]


In [14]:
print("Host:", socket.gethostname())


Host: DGX-H200-USFQ


In [15]:
pipeline = RagPipeline(
    collection="mmia6013_rag_baseline"
)

print("Backend:", os.environ["EMBEDDING_BACKEND"])
print("Modelo:", pipeline.encoder.modelo if hasattr(pipeline.encoder, "modelo") else "bge-m3")
print("Tope del modelo:", pipeline.max_seq_tokens)
print("Colección:", pipeline.collection)

/opt/datadog-packages/datadog-apm-library-python/4.11.1/ddtrace_pkgs/site-packages-ddtrace-py3.12-manylinux2014-x86_64/ddtrace/internal/module.py:314: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  self.loader.exec_module(module)


Backend: h200
Modelo: bge-m3
Tope del modelo: 8192
Colección: mmia6013_rag_baseline


#### 1. Ingesta. Parsea los documentos (PDF → texto). Documenta qué se pierde o se degrada (tablas, encabezados,pies de página) y qué documentos, si alguno, la ingesta rechazó por no tener texto útil.

In [16]:
corpus_dir = ROOT / "corpus"

print("Corpus:", corpus_dir)
print("¿Existe?:", corpus_dir.exists())
print("¿Es carpeta?:", corpus_dir.is_dir())

documentos = sorted(corpus_dir.glob("*"))

print("\nDocumentos encontrados:")
for doc in documentos:
    print(" -", doc.name)

Corpus: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/corpus
¿Existe?: True
¿Es carpeta?: True

Documentos encontrados:
 - .gitkeep
 - bai-2022-constitutional-ai.pdf
 - brown-2020-gpt3.pdf
 - ouyang-2022-instructgpt.pdf
 - rafailov-2023-dpo.pdf
 - vaswani-2017-attention-is-all-you-need.pdf
 - wei-2022-chain-of-thought.pdf


In [17]:
chunks = pipeline.ingest(
    corpus_dir,
    chunk_tokens=512,
    overlap_tokens=102
)

print(f"\nFragmentos obtenidos: {len(chunks)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (30809 > 8192). Running this sequence through the model will result in indexing errors


ingesta: bai-2022-constitutional-ai.pdf: 34 página(s), 94294 caracteres útiles → 75 fragmento(s)
ingesta: brown-2020-gpt3.pdf: 75 página(s), 186342 caracteres útiles → 156 fragmento(s)
ingesta: ouyang-2022-instructgpt.pdf: 68 página(s), 143167 caracteres útiles → 118 fragmento(s)
ingesta: rafailov-2023-dpo.pdf: 27 página(s), 71700 caracteres útiles → 66 fragmento(s)
ingesta: vaswani-2017-attention-is-all-you-need.pdf: 15 página(s), 31566 caracteres útiles → 25 fragmento(s)
ingesta: wei-2022-chain-of-thought.pdf: 43 página(s), 105503 caracteres útiles → 90 fragmento(s)

Fragmentos obtenidos: 530


In [ ]:


datos_chunks = []

for chunk in chunks:
    datos_chunks.append({
        "chunk_id": chunk.id,
        "documento": chunk.document,
        "chunk_index": chunk.chunk_index,
        "tokens": len(
            pipeline.encoder.tokenizer(
                chunk.text,
                add_special_tokens=False,
                truncation=False
            )["input_ids"]
        ),
        "texto": chunk.text
    })

df_chunks = pd.DataFrame(datos_chunks)
print("Total chunks:", len(df_chunks))

,chunk_id,documento,chunk_index,tokens,texto
0,bai-2022-constitutional-ai-0000,bai-2022-constitutional-ai.pdf,0,512,[page=1] Constitutional AI: Harmlessness from ...
1,bai-2022-constitutional-ai-0001,bai-2022-constitutional-ai.pdf,1,512,"from the finetuned model, use a model to evalu..."
2,bai-2022-constitutional-ai-0002,bai-2022-constitutional-ai.pdf,2,510,"(RL) stage, shown as the sequence of steps at ..."
3,bai-2022-constitutional-ai-0003,bai-2022-constitutional-ai.pdf,3,512,"ying a general AI system, we cannot avoid choo..."
4,bai-2022-constitutional-ai-0004,bai-2022-constitutional-ai.pdf,4,512,+ Harmless Figure 2 We show harmlessness versu...


Total chunks: 530


In [20]:
df_chunks.head()

,chunk_id,documento,chunk_index,tokens,texto
0,bai-2022-constitutional-ai-0000,bai-2022-constitutional-ai.pdf,0,512,[page=1] Constitutional AI: Harmlessness from ...
1,bai-2022-constitutional-ai-0001,bai-2022-constitutional-ai.pdf,1,512,"from the finetuned model, use a model to evalu..."
2,bai-2022-constitutional-ai-0002,bai-2022-constitutional-ai.pdf,2,510,"(RL) stage, shown as the sequence of steps at ..."
3,bai-2022-constitutional-ai-0003,bai-2022-constitutional-ai.pdf,3,512,"ying a general AI system, we cannot avoid choo..."
4,bai-2022-constitutional-ai-0004,bai-2022-constitutional-ai.pdf,4,512,+ Harmless Figure 2 We show harmlessness versu...


In [21]:
carpeta_resultados = ROOT / "resultados" / "parte1"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo_chunks = carpeta_resultados / "numeral1.csv"

df_chunks.to_csv(
    archivo_chunks,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_chunks}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral1.csv


### 2. Fragmentación. Baseline de tamaño fijo con solapamiento, en tokens del modelo de embeddings, con `chunk_tokens` y  `overlap_tokens` explícitos. El tope de bge-m3 es 8192 y el  Lab-02 del día miércoles corta por defecto en 512: la justificación dice por qué tu tamaño sirve a tus preguntas, no solo que cabe.

In [22]:
CHUNK_TOKENS = 512
OVERLAP_TOKENS = 102

print("Parámetros de fragmentación")
print("chunk_tokens:", CHUNK_TOKENS)
print("overlap_tokens:", OVERLAP_TOKENS)
print("máximo bge-m3:", pipeline.max_seq_tokens)

Parámetros de fragmentación
chunk_tokens: 512
overlap_tokens: 102
máximo bge-m3: 8192


Se utiliza un tamaño de 512 tokens con 102 tokens de solapamiento. Este tamaño sigue el baseline propuesto por el Lab-02 y mantiene los fragmentos suficientemente por debajo del límite de 8192 tokens de bge-m3. El solapamiento permite conservar parte del contexto cuando una idea se encuentra cerca del límite entre dos fragmentos. La elección será evaluada posteriormente mediante el golden set, por lo que no se asume de antemano que 512 sea óptimo.

###  3. Embedding e indexación. Con bge-m3 en la H200 (EMBEDDING_BACKEND=h200, el valor por defecto, con GlobalProtect conectada) o, sin VPN, con OpenAI (EMBEDDING_BACKEND=openai). Declara en el informe la fila por su id y con su verified_at. Indexa en Qdrant (el contenedor, no :memory:, que es solo para depurar).

In [23]:
pipeline.index(chunks)

print("Índice creado correctamente.")
print("Colección:", pipeline.collection)
print("Número de chunks indexados:", len(chunks))

Índice creado correctamente.
Colección: mmia6013_rag_baseline
Número de chunks indexados: 530


In [25]:
info = cliente_qdrant.get_collection(
    collection_name=pipeline.collection
)

print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=530 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prev

In [26]:
cantidad = cliente_qdrant.count(
    collection_name=pipeline.collection,
    exact=True
)

print("Vectores en Qdrant:", cantidad.count)

Vectores en Qdrant: 530


In [27]:
df_embedding = pd.DataFrame([{
    "id": "embed_local_multilingue",
    "proveedor": "BAAI",
    "modelo": "BAAI/bge-m3",
    "dimension": 1024,
    "tokens_secuencia": 8192,
    "multilingue": True,
    "costo_usd_1M": 0,
    "verified_at": "2026-08-27"
}])

display(df_embedding)

,id,proveedor,modelo,dimension,tokens_secuencia,multilingue,costo_usd_1M,verified_at
0,embed_local_multilingue,BAAI,BAAI/bge-m3,1024,8192,True,0,2026-08-27


In [28]:
archivo_embedding = carpeta_resultados / "numeral3_embedding_baseline.csv"

df_embedding.to_csv(
    archivo_embedding,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_embedding}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral3_embedding_baseline.csv


### 4. Recuperación. Muestra los top-5 fragmentos para al menos 3 preguntas de prueba, con documento y puntaje.

In [29]:
preguntas_prueba = [
    "PREGUNTA 1",
    "PREGUNTA 2",
    "PREGUNTA 3",
]

In [30]:
resultados_retrieval = []

for pregunta in preguntas_prueba:

    resultados = pipeline.retrieve(
        pregunta,
        top_k=5
    )

    print("=" * 100)
    print("PREGUNTA:")
    print(pregunta)
    print("\nTOP 5 VECINOS:\n")

    for rank, resultado in enumerate(resultados, start=1):

        print(f"--- Vecino {rank} ---")
        print(f"Score:     {resultado['score']:.4f}")
        print(f"Documento: {resultado['document']}")
        print(f"Chunk ID:  {resultado['chunk_id']}")
        print(f"Texto:    {resultado['text']}")
        print()

        resultados_retrieval.append({
            "pregunta": pregunta,
            "rank": rank,
            "score": resultado["score"],
            "documento": resultado["document"],
            "chunk_id": resultado["chunk_id"],
            "texto": resultado["text"]
        })

PREGUNTA:
PREGUNTA 1

TOP 5 VECINOS:

--- Vecino 1 ---
Score:     0.4885
Documento: rafailov-2023-dpo.pdf
Chunk ID:  rafailov-2023-dpo-0058
Texto:    XTREMELY anxious/nervous. We have talked about this for months, and I know she will say yes, but I can’t seem to relax. The other day I bought the ring, which I fell in love with myself and know she will too. And then two days ago, I asked her dad’s blessing. Everything has gone great so far, but now leading up to it I can’t think about anything else. I can’t sleep. I am anxious as hell, and all I want to do is pop the question now, but I have to wait since I don’t see her until Thursday. I know how I am going to do it. I have thoroughly planned it out and I am more than prepared. But, at the same time, I am freaking out. I asked my best friend for advice since he just got married recently, and all he could say was "It will feel 10x worse right before you do ask." Any suggestions on how to relax leading up to it? I would like to be able t

In [31]:
df_retrieval = pd.DataFrame(resultados_retrieval)

display(df_retrieval)

,pregunta,rank,score,documento,chunk_id,texto
0,PREGUNTA 1,1,0.488457,rafailov-2023-dpo.pdf,rafailov-2023-dpo-0058,XTREMELY anxious/nervous. We have talked about...
1,PREGUNTA 1,2,0.472186,ouyang-2022-instructgpt.pdf,ouyang-2022-instructgpt-0093,"was hungry. Question: In the passage above, wh..."
2,PREGUNTA 1,3,0.439153,brown-2020-gpt3.pdf,brown-2020-gpt3-0127,16 BLEU-sb test 64 2.39 3.08 3.49 3.56 16.8 8....
3,PREGUNTA 1,4,0.437450,brown-2020-gpt3.pdf,brown-2020-gpt3-0128,23.6 26.1 30.3 33.3 35.5 39.9 De→En 16 BLEU-mb...
4,PREGUNTA 1,5,0.433050,ouyang-2022-instructgpt.pdf,ouyang-2022-instructgpt-0092,high. • a woman is standing next to an ocean a...
5,PREGUNTA 2,1,0.511419,rafailov-2023-dpo.pdf,rafailov-2023-dpo-0058,XTREMELY anxious/nervous. We have talked about...
6,PREGUNTA 2,2,0.478913,ouyang-2022-instructgpt.pdf,ouyang-2022-instructgpt-0093,"was hungry. Question: In the passage above, wh..."
7,PREGUNTA 2,3,0.456087,ouyang-2022-instructgpt.pdf,ouyang-2022-instructgpt-0092,high. • a woman is standing next to an ocean a...
8,PREGUNTA 2,4,0.451706,ouyang-2022-instructgpt.pdf,ouyang-2022-instructgpt-0067,3}. Desination will be a red barn on the right...
9,PREGUNTA 2,5,0.448072,rafailov-2023-dpo.pdf,rafailov-2023-dpo-0057,"doing chores, keeping my room clean, cleaning ..."


In [32]:
archivo_retrieval = carpeta_resultados / "numeral4_retrieval_top5.csv"

df_retrieval.to_csv(
    archivo_retrieval,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_retrieval}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral4_retrieval_top5.csv
